# ZuCo text-only sentiment baseline

Frozen and fine-tuned BERT baselines for ZuCo Task 1 sentiment, run with
nested stratified cross-validation (held-out test set per fold).

Use a **GPU runtime**: *Runtime -> Change runtime type -> GPU*.

## 1. Get the code

Clones the repo on first run and pulls the latest commit on every run after,
so updates pushed to GitHub show up here by re-running this cell.

In [ ]:
REPO = "https://github.com/parmisbathaeiyan/zuco-text-baseline.git"
NAME = "zuco-text-baseline"

import os
if not os.path.exists(NAME):
    !git clone -q $REPO
else:
    !git -C $NAME pull -q
%cd $NAME

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Pooling and where to save results

`POOLING` selects how token embeddings are reduced to a sentence vector:
`mean` (masked average, the default) or `cls` (the `[CLS]` token). Set it once
here; each value writes to its own `OUTPUT_DIR` so the two never overwrite.

JSON summaries are written to `OUTPUT_DIR`; all plots (confusion matrices,
learning curves, the comparison chart) go to `OUTPUT_DIR/plots`.

Mounting Drive is optional but recommended: the local Colab disk is wiped when
the runtime disconnects, whereas a Drive folder keeps your results. To stay on
local disk instead, just set `OUTPUT_DIR = "results"` and skip the mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Pooling over the token embeddings: 'mean' (masked average) or 'cls'.
# Each pooling writes to its own folder so the two sets never mix.
POOLING = 'cls'

BASE_DIR = '/content/drive/MyDrive/Parmis/Thesis/Results/zuco_text_baseline_again'
folder_name = 'zuco_results_v2'
OUTPUT_DIR = f'{BASE_DIR}/{folder_name}' if POOLING == 'mean' else f'{BASE_DIR}/{folder_name}_{POOLING}'
print('pooling:', POOLING, '->', OUTPUT_DIR)
# OUTPUT_DIR = 'results'  # uncomment to keep results on the local Colab disk

## 4. Frozen baseline

Encoder weights stay fixed; only a linear probe on top of the pooled
features is trained.

In [ ]:
!python run.py --mode frozen --model-name bert-base-uncased \
    --pooling "$POOLING" --output-dir "$OUTPUT_DIR"

## 5. Fine-tuned baseline

The whole encoder is fine-tuned. This is the real text-only ceiling.

In [ ]:
!python run.py --mode finetune --model-name bert-base-uncased \
    --pooling "$POOLING" --output-dir "$OUTPUT_DIR"

## 6. (Optional) sweep a few backbones

RoBERTa often edges out BERT on sentiment; DistilBERT is the quick option.

In [ ]:
!python run.py --mode both --pooling "$POOLING" --output-dir "$OUTPUT_DIR" \
    --model-name bert-base-uncased roberta-base distilbert-base-uncased

## 7. Multilingual backbones

Multilingual BERT and LaBSE, both frozen and fine-tuned. They go through
the same mean-pooled head as the others, so the numbers sit on the same
scale as the runs above.

In [ ]:
!python run.py --mode both --pooling "$POOLING" --output-dir "$OUTPUT_DIR" \
    --model-name bert-base-multilingual-cased sentence-transformers/LaBSE

## 8. Plots for this pooling

Three layers, from overview to detail:

1. **comparison.png** - every backbone's test accuracy and macro-F1, frozen vs
   fine-tuned, in one chart.
2. **curves_by_model_<setup>.png** - one chart per setup overlaying every
   backbone's test loss / accuracy / macro-F1 against epoch.
3. **`<setup>_<model>_curves.png`** - the full train / val / test learning curve
   for each individual run.

Re-run after adding more results.

In [ ]:
!python plot_results.py --results-dir "$OUTPUT_DIR"

In [ ]:
import glob
from IPython.display import Image, display, Markdown

plot_dir = f'{OUTPUT_DIR}/plots'

display(Markdown('### Backbone comparison (test accuracy & macro-F1)'))
display(Image(f'{plot_dir}/comparison.png'))

display(Markdown('### Test curves by backbone, per setup'))
for path in sorted(glob.glob(f'{plot_dir}/curves_by_model_*.png')):
    display(Image(path))

display(Markdown('### Per-run learning curves (train / val / test)'))
for path in sorted(glob.glob(f'{plot_dir}/*_curves.png')):
    display(Image(path))

## 9. Confusion matrices

The per-run confusion matrices, saved under `OUTPUT_DIR/plots`.

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob(f'{OUTPUT_DIR}/plots/*_confusion.png')):
    display(Image(path))

## 10. Score table

Averaged test scores from every saved run.

In [ ]:
import json, glob
for path in sorted(glob.glob(f'{OUTPUT_DIR}/*.json')):
    s = json.load(open(path))
    print(f"{s['mode']:<9} {s.get('pooling', 'mean'):<5} {s['model_name']:<32} "
          f"acc {s['accuracy_mean']:.3f}+/-{s['accuracy_std']:.3f}  "
          f"f1 {s['macro_f1_mean']:.3f}+/-{s['macro_f1_std']:.3f}")

## 11. Compare pooling: mean vs cls

Needs **both** folders to exist, so run the whole notebook once with
`POOLING = 'mean'` and once with `POOLING = 'cls'` first. This then builds, into
a `_compare_pooling` folder:

- **scores_accuracy.png / scores_macro_f1.png** - grouped bars per setup, each
  backbone's mean vs cls side by side
- **delta_macro_f1.png** - cls minus mean, so positive bars mean cls wins
- **curves_<setup>.png** - test macro-F1 vs epoch, colour = backbone,
  line style = pooling

In [ ]:
MEAN_DIR = f'{BASE_DIR}/{folder_name}'
CLS_DIR = f'{BASE_DIR}/{folder_name}_cls'
COMPARE_DIR = f'{BASE_DIR}/{folder_name}_compare_pooling'

!python compare_runs.py --dirs "$MEAN_DIR" "$CLS_DIR" --labels mean cls --out "$COMPARE_DIR"

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob(f'{COMPARE_DIR}/*.png')):
    display(Image(path))